### 파일 불러오기

In [65]:
import sys
from pathlib import Path

# 프로젝트 루트: notebooks/ 하위 어느 깊이에서 실행해도 동작
_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "code" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "code"))
        break
else:
    raise FileNotFoundError("code/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
from project_paths import DATA_CAFE_ONLY, DATA_INTEGRATED, CONFIG_STOPWORDS

pd.set_option('display.max_colwidth', 150)

In [102]:
# 파일 불러오기

import pandas as pd
import re
import json

# 1. JSON 파일 로드
# 각 플랫폼별로 크롤링된 데이터를 데이터프레임으로 불러옵니다.
df_cafe = pd.read_json(DATA_CAFE_ONLY / '의대증원_카페_v2.json')

print(f"네이버 카페 데이터 개수: {len(df_cafe)}")

df_cafe.head(50)

네이버 카페 데이터 개수: 8517


,title,doc,like,comment_cnt,comment_list,img,div,ch,ch2,time,query_from,query_to
0,자사고 1.59 수시카드,None,None,None,[],0,0,naver,cafe,None,20250630,20250629
1,"격동의 최상위권 입시, 흐름을 읽어야(2025수시)",None,2,0,[],4,0,naver,cafe,2025.06.30. 04:48,20250630,20250629
2,2026학년도 의과대학 수시모집 교과 일반전형 (3),"​인제대는 작년 내신성적을 반영할 때 국영수+과학 상위 2과목만 반영하는 산식으로 다른 학교와는 달리 내신 반영 방식이 다소 특이한 편이었습니다. 이 때문에, 다른 학교와 직접적인 비교는 어려울 수 있으나​대학어디가에 공지된 작년 기준 70%컷인 인제대식 1.0...",16,6,"[{'content': '잘 읽어보았습니다. 한눈에 전체 내용이 잘 들어옵니다. 부연 설명까지 감사합니다~', 'comment_date': '2025.06.29. 10:27', 'like': 1}, {'content': '자신을 객관화 하기도 쉽지 않은데 3인칭...",1,0,naver,cafe,2025.06.29. 10:11,20250630,20250629
3,추가 의대증원 이야기 나오던데..,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요?어디서 본거 같은데 검색하면 또 안나와가지구요 ​​​​​​​​[7월 추천강의] *생물 노용관_이론+기출火木金13:10 https://vo.la/kBEoP​최성윤_기출변형土日13:30 https:...,0,1,"[{'content': '올해는 일단 유예됐는데 취소가 아니라 유예된거라 어떻게 될지 모르겠더라구요,, 회의를 해서 내년에 증원될지 안될지 결정하나봐요', 'comment_date': '2025.06.30. 23:44', 'like': 0}]",0,0,naver,cafe,2025.06.30. 22:02,20250630,20250629
4,(공유)2026 한림대 의예과 모집 현황 및 전년도 입시 결과,None,None,None,[],0,0,naver,cafe,None,20250630,20250629
5,"연세대 미래캠퍼스 의예과 3년, 12년특례 면접준비 전략","KGS 메가반 카페입니다. ​연세대 미래캠 의예과는 1단계 서류 100% + 2단계 면접 40% 비중으로 최종 합격자가 결정됩니다.면접은 단순한 구술 확인이 아니라 ‘의대 인재상에 맞는 가치관, 사고력, 전공 적합성’을 확인하는 결정적 평가 요소입니다.​2025...",6,0,[],3,0,naver,cafe,2025.06.30. 18:21,20250630,20250629
6,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 유리한가요?올해 재수생도 많고 의대 증원도 폐지된 거 감안하면 저 정도면 sky 불안할까요?ㅠㅠ​현역입니당,0,3,"[{'content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ㅈㄴ 떨림 지금 스카이중에 유리한거 생각할 이유는 없다 성적 나오는거 보고 유리한거 골라가면됨', 'comment_date': '2025.06.29. 22:24', 'like': 0}, {'co...",0,0,naver,cafe,2025.06.29. 22:22,20250630,20250629
7,이준석 욕하는 새끼들은 양심이 뒤진거지,이준석 스스로 탈당햇나(x)​국힘당과 노인은 물론 강제로 나가라고 했잖아.​강제로 내쫓고 창당해도 망할꺼라고 ㅋㅋ​ 둔스톤 나가면 선거다이긴다고 니들이 틀튜브 보면서 그지랄발광떨었잔어 그래놓고 왜 이준석탓이야. 확실하지도 않은 성상납으로 모욕주고 내칠땐 언제...,1,0,[],1,0,naver,cafe,2025.06.29. 11:54,20250630,20250629
8,"[기사 공유] 작년 대입 결과, 어떻게 해석할까","​입시 초보맘인데, 올해 황금돼지 인원 많다, 의대 증원 백지화 등 변화가 너무 커 과거 입결을 어떻게 봐야할지 너무 고민됩니다. 제발 배치 컨설팅에서 궁금증이 해소되길 기대합니다. . . ㅠ.ㅠ​​​지난 입시 결과 활용 전략 기사 요약​대학별 산정 기준 확인:...",3,0,[],1,0,naver,cafe,2025.06.30. 21:41,20250630,20250629
9,수시 6장... 머리터지기 시작했어요,"게시판 안내를 확인해 주세요! ※ 글쓰기 전 선검색 바랍니다 ※ 미취학글, 과외구인글, 아이디 추천글, 댓글 달린 글 삭제시 준회원강등 및 (최하7일간)활동정지 ※ 주의 사항 및 카페 규정 반드시 숙지 ▶ https://cafe.naver.com/mathall...",1,11,"[{'content': '전체내신나오고, 아이의 생기부가 어느정도 파악되면 객관적인 아이의 현재 위치에 맞게 아이의 성향(종합,교과,논술,정시)에 따라 학교,학과를 20개 15개,10개 좁혀가면서 제작년의 입시결과를 참고하시면 좋을듯해요. 작년에는 의대증원으로 ...",0,0,naver,cafe,2025.06.30. 23:28,20250630,20250629


### 본문, 댓글 비어있는 행 제거

In [103]:
# 본문, 댓글 비어있는 행 제거
import pandas as pd

# apply 함수를 사용하여 각 행의 리스트 길이를 체크
zero_comment_list_cnt = df_cafe[df_cafe['comment_list'].apply(len) == 0].shape[0]
print(f"댓글 리스트가 비어있는 데이터 개수: {zero_comment_list_cnt}개")

empty_docs = df_cafe[df_cafe['doc'].isnull() | (df_cafe['doc'] == '')]

# 2. 개수 확인
print(f"본문이 비어있는 데이터 개수: {len(empty_docs)}개")


# 1. 본문도 있고 댓글 리스트도 있는 데이터만 필터링
# 조건 1: doc가 null이 아님 (notnull)
# 조건 2: doc가 빈 문자열이 아님 (!= '')
# 조건 3: comment_list의 길이가 0보다 큼 (apply len > 0)
df_filtered = df_cafe[
    (df_cafe['doc'].notnull()) & 
    (df_cafe['doc'].str.strip() != '') & 
    (df_cafe['comment_list'].apply(len) > 0) 
].copy()

# 2. 인덱스 재정렬
df_filtered.reset_index(drop=True, inplace=True)

# 3. 결과 확인
print(f"✅ 최종 필터링된 데이터 개수: {len(df_filtered)}개")
print(f"🗑️ 제거된 데이터(본문누락 or 댓글없음): {len(df_cafe) - len(df_filtered)}개")

댓글 리스트가 비어있는 데이터 개수: 3075개
본문이 비어있는 데이터 개수: 3025개
✅ 최종 필터링된 데이터 개수: 3762개
🗑️ 제거된 데이터(본문누락 or 댓글없음): 4755개


### 인덱스 재정렬

In [104]:
# 인덱스 재정렬
# 제목: title
# 본문: doc
# 페이지 좋아요 수: like
# 댓글 수: comment_cnt
# 댓글 리스트: comment_list
# comment_list = [{'comment_content': 댓글 내용,
#                  'comment_like': 댓글 공감 수,
#                   'comment_date': 댓글 작성일}]
# 채널 종류(blog/cafe): ch
# 작성일: date(형식: 2025-01-01)
# 구간: section (1-4구간)

import pandas as pd
import ast

# 원본 데이터 복사 (원본 보존용)
df_reindex = df_filtered.copy()

# ---------------------------------------------------------
# 1. 작성일(date) 포맷 변경 및 컬럼 생성
# 기존 'time' 컬럼의 "2025.06.29. 10:11" -> 앞 10자리 슬라이싱 후 '.'을 '-'로 치환
# ---------------------------------------------------------
df_reindex['date'] = df_reindex['time'].astype(str).str.slice(0, 10).str.replace('.', '-')

# ---------------------------------------------------------
# 2. 구간(section) 할당 로직
# ---------------------------------------------------------
# 정확한 비교를 위해 date를 datetime 타입으로 임시 변환
dt_series = pd.to_datetime(df_reindex['date'], errors='coerce')

def get_section(date_val):
    if pd.isna(date_val): return None
    if pd.Timestamp('2024-01-01') <= date_val <= pd.Timestamp('2024-03-31'): return 1
    elif pd.Timestamp('2024-04-01') <= date_val <= pd.Timestamp('2024-06-30'): return 2
    elif pd.Timestamp('2024-07-01') <= date_val <= pd.Timestamp('2024-12-31'): return 3
    elif pd.Timestamp('2025-01-01') <= date_val <= pd.Timestamp('2025-06-30'): return 4
    else: return 0 # 예외 구간 (설정하신 기간 외의 데이터)

df_reindex['section'] = dt_series.apply(get_section)

# ---------------------------------------------------------
# 3. comment_list 딕셔너리 키 및 날짜 포맷 수정
# ---------------------------------------------------------
def process_comments(comments_data):
    # 1. 아예 비어있는 리스트인 경우 안전하게 빈 리스트 반환
    if isinstance(comments_data, list):
        if len(comments_data) == 0:
            return []
        comments = comments_data
        
    # 2. 결측치(NaN)인 경우 (float 타입이면서 NaN일 때만 걸러냄)
    elif isinstance(comments_data, float) and pd.isna(comments_data):
        return []
        
    # 3. 문자열인 경우 (CSV에서 불러와서 '[]' 처럼 텍스트로 굳어있는 경우)
    elif isinstance(comments_data, str):
        # 텍스트 자체가 비어있거나 대괄호만 있는 경우
        if comments_data.strip() in ["", "[]", "NaN", "nan"]:
            return []
        try:
            # 문자열을 실제 리스트/딕셔너리 객체로 변환
            comments = ast.literal_eval(comments_data)
        except (ValueError, SyntaxError):
            return []
            
    # 4. 그 외의 예외적인 형태가 들어오면 빈 리스트 반환
    else:
        return []
        
    # --- 여기서부터는 정상적인 리스트 형태의 데이터 정제 ---
    new_comments = []
    for c in comments:
        # 가끔 딕셔너리가 아닌 이상한 값이 섞여 있을 때를 대비한 방어 코드
        if not isinstance(c, dict):
            continue
            
        raw_date = str(c.get('comment_date', ''))
        formatted_date = raw_date[:10].replace('.', '-') if raw_date else ''
        
        new_dict = {
            'comment_content': c.get('content', ''),
            'comment_like': c.get('like', 0),
            'comment_date': formatted_date
        }
        new_comments.append(new_dict)
        
    return new_comments

df_reindex['comment_list'] = df_reindex['comment_list'].apply(process_comments)

# ---------------------------------------------------------
# 4. 컬럼 삭제 및 이름 변경 (인덱스 정리)
# ---------------------------------------------------------
# 삭제 대상: ch(기존), img, div, query_from, query_to, time
cols_to_drop = ['ch', 'img', 'div', 'query_from', 'query_to', 'time']
df_reindex = df_reindex.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

# ch2를 ch로 변경
df_reindex = df_reindex.rename(columns={'ch2': 'ch'})

# 최종적으로 원하는 컬럼 순서로 정렬 (선택 사항)
final_cols = ['title', 'doc', 'like', 'comment_cnt', 'comment_list', 'ch', 'date', 'section']
df_reindex = df_reindex[final_cols]

print("✅ 데이터 전처리 및 구조화 완료!")

✅ 데이터 전처리 및 구조화 완료!


In [105]:
df_reindex.head(5)

,title,doc,like,comment_cnt,comment_list,ch,date,section
0,2026학년도 의과대학 수시모집 교과 일반전형 (3),"​인제대는 작년 내신성적을 반영할 때 국영수+과학 상위 2과목만 반영하는 산식으로 다른 학교와는 달리 내신 반영 방식이 다소 특이한 편이었습니다. 이 때문에, 다른 학교와 직접적인 비교는 어려울 수 있으나​대학어디가에 공지된 작년 기준 70%컷인 인제대식 1.0...",16,6,"[{'comment_content': '잘 읽어보았습니다. 한눈에 전체 내용이 잘 들어옵니다. 부연 설명까지 감사합니다~', 'comment_like': 1, 'comment_date': '2025-06-29'}, {'comment_content': '자신을 객...",cafe,2025-06-29,4
1,추가 의대증원 이야기 나오던데..,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요?어디서 본거 같은데 검색하면 또 안나와가지구요 ​​​​​​​​[7월 추천강의] *생물 노용관_이론+기출火木金13:10 https://vo.la/kBEoP​최성윤_기출변형土日13:30 https:...,0,1,"[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된거라 어떻게 될지 모르겠더라구요,, 회의를 해서 내년에 증원될지 안될지 결정하나봐요', 'comment_like': 0, 'comment_date': '2025-06-30'}]",cafe,2025-06-30,4
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 유리한가요?올해 재수생도 많고 의대 증원도 폐지된 거 감안하면 저 정도면 sky 불안할까요?ㅠㅠ​현역입니당,0,3,"[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ㅈㄴ 떨림 지금 스카이중에 유리한거 생각할 이유는 없다 성적 나오는거 보고 유리한거 골라가면됨', 'comment_like': 0, 'comment_date': '2025-06-2...",cafe,2025-06-29,4
3,수시 6장... 머리터지기 시작했어요,"게시판 안내를 확인해 주세요! ※ 글쓰기 전 선검색 바랍니다 ※ 미취학글, 과외구인글, 아이디 추천글, 댓글 달린 글 삭제시 준회원강등 및 (최하7일간)활동정지 ※ 주의 사항 및 카페 규정 반드시 숙지 ▶ https://cafe.naver.com/mathall...",1,11,"[{'comment_content': '전체내신나오고, 아이의 생기부가 어느정도 파악되면 객관적인 아이의 현재 위치에 맞게 아이의 성향(종합,교과,논술,정시)에 따라 학교,학과를 20개 15개,10개 좁혀가면서 제작년의 입시결과를 참고하시면 좋을듯해요. 작년에는...",cafe,2025-06-30,4
4,고3 수시 대학 라인 문제,게시판 안내를 확인해 주세요! 🚩 고1~고3 필수! 학종 합격 가이드 👉 https://bit.ly/4c270aT\n \n \n \n \n \n \n ...,1,2,"[{'comment_content': '같은 고민 입니다..어렵네요ㅠㅠ', 'comment_like': 0, 'comment_date': '2025-07-02'}, {'comment_content': '안녕하세요. 회원님유니브클래스 카페 회원이 된 것을 축하드립...",cafe,2025-06-30,4


### 텍스트 기본 정제

In [108]:
# 텍스트 기본 정제
# 불필요한 노이즈 제거

# 순서의 중요성: 반드시 텍스트 정제(HTML, 공백 제거 등)를 먼저 진행한 후 유사도 80% 검사를 해야 합니다. 공백이나 태그가 섞여 있으면 똑같은 내용의 글도 컴퓨터는 다른 글로 인식하기 때문입니다.
# SequenceMatcher 최적화: 2만 개의 본문을 모든 글과 1:1로 비교하면 연산량이 너무 많아 주피터가 뻗을 수 있습니다. 그래서 **groupby('title')**을 사용하여 "우선 제목이 같은 글들끼리만" 본문 80% 유사도 검사를 하도록 범위를 좁혀 연산 속도를 대폭 높였습니다. (AI 개발에서 중요한 시간 복잡도 최적화입니다.)
# 댓글 보존: ㅋㅋ, ㅎㅎ 같은 자음만으로 이루어진 무의미한 댓글은 clean_text를 거치면 빈 문자열("")이 됩니다. if cleaned_content: 조건을 통해 이러한 쓰레기 데이터를 리스트에서 자연스럽게 날려버렸습니다.

import re
from difflib import SequenceMatcher

# ---------------------------------------------------------
# 1. 텍스트 정제 함수 정의 (조건 1, 2, 4 통합)
# ---------------------------------------------------------
import re
import pandas as pd

def clean_text(text):
    if pd.isna(text): 
        return ""
    
    text = str(text)
    
    # 1. 특정 문장 제거 (네이버 카페 기본 양식)
    text = re.sub(r'게시판 안내를 확인해\s*주세요!?', '', text)
    text = re.sub(r'이해원연구소 코어패스 인터그레이트 4월 문항 공모!?', '', text)
    text = re.sub(r'실거래가 게시글에 관한 의견을 묻습니다!?', '', text)
    text = re.sub(r'수만휘는 수험생 대학생 선생님 학부모님이 함께 모여 서로 돕고 응원하는 대한민국 최대의 대입 커뮤니티입니다 선배들의 따뜻한 조언과 회원들의 존중 속에서 함께 만들어가는 공간입니다 이용 규정 보기!?','', text)
    
    # (선택) 추가로 자주 나오는 양식문구도 이곳에 넣을 수 있습니다.
    # text = re.sub(r'※\s*미취학글,\s*과외구인.*', '', text) 
    
    # 2. 영문 시작 ~ com으로 끝나는 도메인/URL 제거
    text = re.sub(r'[a-zA-Z][a-zA-Z0-9\.\-]*com\b', '', text)
    
    # 3. 일반적인 http/https URL 제거
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 4. HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    
    # 5. 자음/모음 반복 (ㅋㅋㅋ, ㅎㅎㅎ, ㅠㅠㅠ 등) 제거
    text = re.sub(r'[ㄱ-ㅎㅏ-ㅣ]+', '', text)
    
    # 6. 특수문자 및 이모지 제거 (한글, 영문, 숫자, 공백만 남김)
    # 주의: 여기서 점(.)이 날아가기 때문에 반드시 도메인 제거(2번)를 이보다 먼저 해야 합니다.
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    
    # 7. 연속된 공백, \n, \t 등을 하나의 공백으로 치환하고 양쪽 여백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# (선택) 댓글에도 .com 링크가 있을 수 있으니 댓글도 다시 정제
# df['comment_list'] = df['comment_list'].apply(clean_comments_list) 


# ---------------------------------------------------------
# 2. comment_list 내부 텍스트 정제 함수
# ---------------------------------------------------------
def clean_comments_list(comments):
    # 1. 만약 리스트가 아니라 문자열('[...]' 형태)로 굳어있다면 리스트로 변환
    if isinstance(comments, str):
        try:
            comments = ast.literal_eval(comments)
        except (ValueError, SyntaxError):
            return comments # 변환 실패 시 원본 유지
            
    # 리스트가 아니면 그대로 반환
    if not isinstance(comments, list):
        return comments
    
    cleaned_comments = []
    for c in comments:
        if not isinstance(c, dict):
            continue
            
        # 2. [핵심 수정] 키 이름 방어 로직
        # 'comment_content'를 먼저 찾고, 없으면 원본 키인 'content'를 찾음
        original_text = c.get('comment_content', c.get('content', ''))
        
        # 텍스트 정제 함수 적용
        cleaned_content = clean_text(original_text)
        
        # 정제 후 내용이 남아있는 경우만 보존
        if cleaned_content:
            new_c = c.copy()
            # 현재 딕셔너리가 갖고 있는 키 이름에 맞춰서 업데이트
            if 'comment_content' in new_c:
                new_c['comment_content'] = cleaned_content
            else:
                new_c['content'] = cleaned_content
                
            cleaned_comments.append(new_c)
            
    return cleaned_comments
# ---------------------------------------------------------
# [실행 파트] 정제 함수 적용
# ---------------------------------------------------------

# 정제 함수 적용
df_reindex['title'] = df_reindex['title'].apply(clean_text)
df_reindex['doc'] = df_reindex['doc'].apply(clean_text)
df_reindex['comment_list'] = df_reindex['comment_list'].apply(clean_comments_list)

# 2. 내용이 너무 짧아진(노이즈만 있던) 게시글 1차 필터링
df_reindex = df_reindex[df_reindex['doc'].str.len() >= 5].reset_index(drop=True)

print(f"✅ 1단계 정제 완료!")

# ---------------------------------------------------------
# 3. 고급 중복 제거: 동일 제목 + 본문 90% 이상 일치 (조건 3)
# ---------------------------------------------------------
print("🔍 2단계: 유사도 80% 이상 중복 게시글을 탐색합니다...")

def is_similar(str1, str2, threshold=0.8):
    # SequenceMatcher를 사용해 두 문자열의 유사도 비율(0.0 ~ 1.0) 계산
    return SequenceMatcher(None, str1, str2).ratio() >= threshold

indices_to_drop = set()

# '제목'이 완전히 같은 그룹끼리 묶어서 검사 (속도 최적화)
for title, group in df.groupby('title'):
    if len(group) > 1:
        # 그룹 내 인덱스 목록
        idxs = group.index.tolist()
        
        # 인덱스끼리 1:1 비교
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                idx1, idx2 = idxs[i], idxs[j]
                
                # 이미 삭제 예정인 인덱스는 건너뜀
                if idx2 in indices_to_drop:
                    continue
                
                doc1 = df_reindex.loc[idx1, 'doc']
                doc2 = df_reindex.loc[idx2, 'doc']
                
                # 본문 유사도가 80% 이상이면 나중 글(idx2)을 삭제 대상에 추가
                if is_similar(doc1, doc2, threshold=0.8):
                    indices_to_drop.add(idx2)

# 중복 인덱스 최종 삭제
before_cnt = len(df_reindex)
df_reindex = df_reindex.drop(index=list(indices_to_drop)).reset_index(drop=True)
after_cnt = len(df_reindex)

print(f"✅ 정제 완료!")
print(f"📊 최종 남은 유효 데이터: {after_cnt}개")

✅ 1단계 정제 완료!
🔍 2단계: 유사도 80% 이상 중복 게시글을 탐색합니다...
✅ 정제 완료!
📊 최종 남은 유효 데이터: 3712개


In [109]:
df_reindex.tail(30)

,title,doc,like,comment_cnt,comment_list,ch,date,section
3682,의대증원 왜 못함,대학병원 예약 몇달씩이고 수술도 잡으면 몇개월인데 기피과 늘리고 하면 되는거 아니냐,0,2,"[{'comment_content': '높으신분들이 높나 반발해서', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_content': '진짜 해결할 거면 수가를 올렸지 걍 쇼임', 'comment_li...",cafe,2024-01-08,1
3683,의사가 미래에도,2027시대인재N 기숙 대치 목동 모집 보기 의사가 미래에도 유망하고 선망을 받는 직업일까 의문이 들었음저출산 고령화로 인구는 계속 감소하고 늙은 사람들만 늘어나잖아 근데 의사는 매년 수천명씩 공장 찍어내듯이 만들어 내는데의사는 별 탈 없이 몸만 움직일수 있으면...,1,4,"[{'comment_content': '증원하는 순간 씁쓸하긴 하지만 별 수 없네요', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_content': '증원되면 지금정도 수입은 어렵죠', 'comme...",cafe,2024-01-08,1
3684,공유 이재명 응급이송 뒷말 무성 의대증원 정책까지 흔들,이재명 응급이송 뒷말 무성 의대증원 정책까지 흔들 이재명 응급이송 뒷말 무성 의대증원 정책까지 흔들,7,1,[{'comment_content': '윤정부가 지지율 나락간데에 공공의대 의사증원이 크게 한몫했죠 저도 크게 배신감 느꼈는데 찢이 정책을 멈추게 하네요 참 아이러니한 세상 좌파들이 왜 의료 교육정책은 입다물고 있을까요 저들이 문재인때 못한 걸 윤정부가 알아서 ...,cafe,2024-01-08,1
3685,의대 증원하면 레지TO도 그만큼 느는 건가요,갑자기 궁금해졌는데,0,6,"[{'comment_content': '수도권은 줄이고 지방만늘듯 지금도 수도권 계속줄이는중', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_content': '는다 해도 기피과만 늘겠죠', 'com...",cafe,2024-01-08,1
3686,이재명 때문에 이득 보는 건 의사들이 아니라 윤석열임,윤돼지 용산놈들은 의대 증원 지르긴 했지만 딱 총선용이었고 원래 보수였던 의사들하고 충돌해서 자칫 의료대란이라도 나기만하면 골치 아프니 적당히 치고 빠지려고 했지만 출구전략이 마땅치 않았음 그런데 찢가놈이 의사들과 자발적으로 척지는 바람에 윤돼지 입장에선 사기치...,8,3,"[{'comment_content': '맞아요특검이 날라가도 사람들은 이재명 욕만 해요블랙홀 핼기사태 입니다 헬기로 헬로 갔어요 아재 개그', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_conten...",cafe,2024-01-08,1
3687,충격 특권충 정청래에 의한 부산대병원의 굴욕,정청래 목은 민감 후유증 고려해 잘하는 곳 병원 에서 해야 부산대병원 비하 논란 더불어민주당 이재명 대표가 2일 부산 가덕도신공항 부지를 방문했다 괴한의 흉기 습격을 받은 것과 관련해 같은 당 소속 정청래 의원의 발언이 역풍에 불을 지피는 모양새다 그가 말로써 ...,10,11,"[{'comment_content': '그래서 내 년 의대증원 1000명 하나요', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_content': '윤통이 빨리 의대 1000명 증원확대 발표해야지요 ...",cafe,2024-01-08,1
3688,의사들은 이재명을 박살내면서 윤석열에 시그널 주는 거지요,총선때까지 이재명 박살내줄테니까 쓸데없이 의대 정원 증설 그거 말꺼내서 적전 분열시키지 말라고 의사들 입장에선 무서운 윤석열과 싸울 사안을 동네 바보 형 같은 이재명과 싸우라니 신바람 난 거지요 119 구급차 타고 서울 갑시다 환자 설득에 진땀 빼는 의료 현장 ...,61,9,"[{'comment_content': '자알했다 재명이 재명이 죄명 하나 더 늘었다 응급의료체계파괴죄', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_content': '이런건 안시켜도 척 보고 알아...",cafe,2024-01-08,1
3689,의대 증원 규모 예상,무기명 투표 중몇명이나 증원할까 500 미만500 9991000 19992000 29993000 이상결과보기 투표하기 투표자 보기투표자 0명확인전체 투표자 보기투표자 0명확인 깔끔하게 1000명 할듯,2,15,"[{'comment_content': '그냥 한 명만 늘리죠', 'comment_like': 0, 'comment_date': '2024-01-08'}, {'comment_content': '저만 들어가게', 'comment_like': 0, 'comment_d...",cafe,2024-01-08,1
3690,의대증원 진짜 하냐,요즘 뉴스 돌아가는거보니까 서울대 의대 증원말고는 하나마나같은데,0,3,"[{'comment_content': '걍 의대증원은 필연임의협이 눕든 말든 여론하고 여야 정치권 대동단결이라 안될 수가 없음', 'comment_like': 0, 'comment_date': '2024-01-07'}, {'comment_content': '하긴...",cafe,2024-01-07,1
3691,북학의 김동은,조선은 무엇이 잘못되었나 인재가 없었나 왕 세종 정조 한글 창제 의정부 설립 금난전권 장용영 국군 설치 수원 화성 건설 초기 조광조 세조 너무 앞서가다가 타이밍이 안맞아서 사형 후기 유형원 인 효 존 로크와 애덤스미스보다도 빨랐던 그의 저서 반계수록 반영X농업과...,0,1,"[{'comment_content': '핵심을 잘 파악하고 정리한 것으로 보이네 너무 압축하면 의미전달이 잘 안될 수 있네 조금 더 풀어써 다른 사람이 바로 이해할수 있게 쓰는게 중요하네 수고했어요', 'comment_like': 0, 'comment_date'...",cafe,2024-01-06,1


In [110]:
# 변환할 숫자형 컬럼 목록
num_cols = ['like', 'comment_cnt', 'section']

for col in num_cols:
    if col in df_reindex.columns:
        # pd.to_numeric을 통해 숫자로 변환 (변환 불가한 값은 NaN 처리 -> 0으로 채움 -> 정수형 변환)
        df_reindex[col] = pd.to_numeric(df_reindex[col], errors='coerce').fillna(0).astype(int)

print("숫자형 데이터 변환이 완료되었습니다.")

숫자형 데이터 변환이 완료되었습니다.


In [112]:
# 최종 데이터 확인
print(f"✅ 최종 전처리 완료 데이터 개수: {len(df_reindex)}개")

# CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
save_path = str(DATA_CAFE_ONLY / '의대증원_cafedata_preprocess.csv')
df_reindex.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"\n데이터가 성공적으로 저장되었습니다: {save_path}")

✅ 최종 전처리 완료 데이터 개수: 3712개

데이터가 성공적으로 저장되었습니다: 의대증원_cafedata_preprocess.csv
